In [1]:
import torch, platform, sys
print("Python:", sys.version)
print("OS:", platform.platform())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))

Python: 3.10.18 (main, Jun  5 2025, 13:14:17) [GCC 11.2.0]
OS: Linux-5.15.0-134-generic-x86_64-with-glibc2.31
Torch: 2.5.1+cu121
CUDA available: True
GPU: Quadro RTX 8000
VRAM (GB): 47.45


In [2]:
# Jupyter에서 실행:
%pip -q install --upgrade pip

# (권장) transformers 최신(main) + accelerate
%pip -q install "git+https://github.com/huggingface/transformers" accelerate pillow opencv-python

# (선택) VRAM 부족하면 4bit 양자화에 필요
%pip -q install bitsandbytes

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [1]:
%pip -q install --upgrade pip
%pip -q install -U "git+https://github.com/huggingface/transformers" accelerate

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import transformers, inspect, os
print(transformers.__version__)
print(os.path.dirname(inspect.getfile(transformers)))

5.0.0.dev0
/home/ui_detect/anaconda3/envs/ym/lib/python3.10/site-packages/transformers


In [3]:
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
import torch

MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16,   # 또는 bf16 가능
)
model.eval()

print("Loaded:", MODEL_ID)

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

Loaded: Qwen/Qwen3-VL-8B-Instruct


In [4]:
from pathlib import Path

# 입력/출력 경로
INPUT_JSONL  = Path("./out_icon_labels.jsonl")
OUTPUT_JSONL = Path("./out_icon_descriptions.jsonl")

# 모델
MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"

# 크롭 파라미터
ICON_PAD_PX    = 2     # bbox에 살짝 패딩
CTX_PAD_SCALE  = 2.0   # bbox 최대변 * scale 만큼 패딩해서 주변 crop

# 생성 파라미터
MAX_NEW_TOKENS = 96
DO_SAMPLE      = False   # 재현성 원하면 False
TEMPERATURE    = 0.2     # DO_SAMPLE=True일 때만 의미

In [5]:
import os, json
import torch
from PIL import Image
from tqdm import tqdm

from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype  = torch.float16  # 48GB면 fp16 충분

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=dtype,
)
model.eval()

print("Loaded:", MODEL_ID)
print("device:", device, "| dtype:", dtype)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

Loaded: Qwen/Qwen3-VL-8B-Instruct
device: cuda | dtype: torch.float16


In [7]:
def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)

def write_jsonl_line(fp, obj):
    fp.write(json.dumps(obj, ensure_ascii=False) + "\n")

In [8]:
def clamp_box_xyxy(box, W, H):
    x1, y1, x2, y2 = [int(v) for v in box]
    x1 = max(0, min(W - 1, x1))
    y1 = max(0, min(H - 1, y1))
    x2 = max(0, min(W - 1, x2))
    y2 = max(0, min(H - 1, y2))
    if x2 <= x1: x2 = min(W - 1, x1 + 1)
    if y2 <= y1: y2 = min(H - 1, y1 + 1)
    return [x1, y1, x2, y2]

def pad_box_xyxy(box, pad_px, W, H):
    x1, y1, x2, y2 = box
    return clamp_box_xyxy([x1 - pad_px, y1 - pad_px, x2 + pad_px, y2 + pad_px], W, H)

def make_icon_and_context_crops(image: Image.Image, bbox_xyxy, icon_pad_px=2, ctx_pad_scale=2.0):
    W, H = image.size
    box = clamp_box_xyxy(bbox_xyxy, W, H)

    # 1) icon crop
    icon_box = pad_box_xyxy(box, icon_pad_px, W, H)
    icon_img = image.crop(tuple(icon_box))

    # 2) context crop: bbox 크기 기반으로 확장
    bw = box[2] - box[0]
    bh = box[3] - box[1]
    pad = int(max(bw, bh) * ctx_pad_scale)
    ctx_box = pad_box_xyxy(box, pad, W, H)
    ctx_img = image.crop(tuple(ctx_box))

    return box, icon_img, ctx_img

In [14]:
@torch.no_grad()
def describe_icon(icon_img: Image.Image, ctx_img: Image.Image) -> str:
    prompt = (
        "You are describing a UI icon for a dataset.\n"
        "You will be given (1) a cropped ICON image and (2) a UI CONTEXT image around it.\n\n"
        "Task:\n"
        "- Write a natural English description (1–2 sentences).\n"
        "- Describe BOTH the visual appearance (shapes, strokes, symbols) and the likely function.\n"
        "- If uncertain, make the best guess using the context.\n\n"
        "Output rules (STRICT):\n"
        "- Output ONLY the description text.\n"
        "- No bullet points, no quotes, no extra commentary.\n"
    )

    messages = [{
        "role": "user",
        "content": [
            {"type": "text",  "text": prompt},
            {"type": "image", "image": icon_img},
            {"type": "image", "image": ctx_img},
            {"type": "text",  "text": "Return only the description text."},
        ],
    }]

    # chat template -> model text
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # encode
    inputs = processor(
        text=[text],
        images=[icon_img, ctx_img],
        return_tensors="pt",
        padding=True,
    )

    # device 이동 (device_map="auto"여도 입력은 cuda로 보내는 게 보통 안전)
    if torch.cuda.is_available():
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

    gen_kwargs = dict(
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=DO_SAMPLE,
    )
    if DO_SAMPLE:
        gen_kwargs["temperature"] = TEMPERATURE

    out_ids = model.generate(**inputs, **gen_kwargs)
    decoded = processor.batch_decode(out_ids, skip_special_tokens=True)[0].strip()

    # 안전장치: 여러 줄이면 마지막 줄만
    return decoded.splitlines()[-1].strip()

In [17]:
from pathlib import Path
import os

# 현재 작업 디렉토리 기준으로 탐색 (필요하면 시작점을 바꿔도 됨)
START_DIR = Path.cwd()

def find_train_root(start: Path) -> Path:
    # dataset/screenspot/images/train 폴더를 아래로 훑어서 찾기
    hits = list(start.rglob("dataset/screenspot/images/train"))
    if not hits:
        raise FileNotFoundError(f"Could not find 'dataset/screenspot/images/train' under: {start}")
    # 여러 개면 가장 짧은(상대경로가 짧은) 것을 선택
    hits = sorted(hits, key=lambda p: len(str(p)))
    return hits[0].resolve()

NEW_IMAGE_ROOT = find_train_root(START_DIR)
print("FOUND NEW_IMAGE_ROOT =", NEW_IMAGE_ROOT)
print("exists:", NEW_IMAGE_ROOT.exists())

FOUND NEW_IMAGE_ROOT = /home/ui_detect/DL_UI_Detection/ui_detection_ym/dataset/screenspot/images/train
exists: True


In [18]:
def resolve_image_path(old_path: str) -> str:
    # 파일명만 추출 (윈도우/유닉스 경로 혼합 대응)
    fname = Path(old_path.replace("\\", "/")).name
    return str((NEW_IMAGE_ROOT / fname).resolve())

In [19]:
import os
from PIL import Image

sample = next(read_jsonl(INPUT_JSONL))
old_img_path = sample["image"]
bbox_xyxy = sample["bbox_xyxy"]

img_path = resolve_image_path(old_img_path)

print("old:", old_img_path)
print("new:", img_path)
print("exists:", os.path.exists(img_path))

assert os.path.exists(img_path), f"Mapped path not found: {img_path}"

img = Image.open(img_path).convert("RGB")
box, icon_img, ctx_img = make_icon_and_context_crops(img, bbox_xyxy, ICON_PAD_PX, CTX_PAD_SCALE)

desc = describe_icon(icon_img, ctx_img)

print("bbox:", box)
print("description:", desc)

old: C:\\Users\\dldna\\icon\\screenspot\\images\\train\pc_00b0c9f6-b24f-4bff-b492-12117b50e63b.png
new: /home/ui_detect/DL_UI_Detection/ui_detection_ym/dataset/screenspot/images/train/pc_00b0c9f6-b24f-4bff-b492-12117b50e63b.png
exists: False


AssertionError: Mapped path not found: /home/ui_detect/DL_UI_Detection/ui_detection_ym/dataset/screenspot/images/train/pc_00b0c9f6-b24f-4bff-b492-12117b50e63b.png

In [23]:
from pathlib import Path

print("NEW_IMAGE_ROOT:", NEW_IMAGE_ROOT)
print("exists:", NEW_IMAGE_ROOT.exists())

# 폴더 안에 파일이 실제로 뭐가 있는지(상위 20개)
print("some files:", [p.name for p in list(NEW_IMAGE_ROOT.iterdir())[:20]])
print("png count:", len(list(NEW_IMAGE_ROOT.glob("*.png"))))

from pathlib import Path

sample = next(read_jsonl(INPUT_JSONL))
old_img_path = sample["image"]
fname = Path(old_img_path.replace("\\", "/")).name

print("fname from jsonl:", fname)

# ✅ images 폴더까지만 잡고 그 아래 전역 탐색
IMAGES_ROOT = Path("/home/ui_detect/DL_UI_Detection/ui_detection_ym/dataset/screenspot/images").resolve()
print("IMAGES_ROOT exists:", IMAGES_ROOT.exists(), IMAGES_ROOT)

hits = list(IMAGES_ROOT.rglob(fname))
print("hits:", len(hits))
for p in hits[:10]:
    print(" -", p)

NEW_IMAGE_ROOT: /home/ui_detect/DL_UI_Detection/ui_detection_ym/dataset/screenspot/images/train
exists: True
some files: ['web_bcce7aec-b36a-42c5-8beb-ead23f5ada2c.png', 'pc_79eaf625-93f5-42a4-977e-71aba5418ff4.png', 'web_c307580d-6e2e-4ffb-bb0d-3e156ec524e3.png', 'pc_3a625c5d-db8a-41fb-b533-16a7c8a4f345.png', 'web_589e44b7-401b-45f7-a223-b93ac59c5506.png', 'web_6e4b8b0d-45fe-4d2a-864a-782e08ae8d21.png', 'pc_db6b6fa2-11cc-4b06-88cb-22c180f07a3b.png', 'web_2345b5ec-6280-470e-9a6d-92a85d8b8873.png', 'pc_f344a1f6-cbf8-4115-9c9b-a9dfd00fbfe7.png', 'pc_733fbb76-a188-4804-a2c6-788acf844d7c.png', 'pc_7e37689a-0657-4856-8adf-72a21ee640c9.png', 'web_21f41383-9618-40bf-92a6-df3f7c1fb251.png', 'web_0092e09d-acc7-4ced-8871-47cb05a0a554.png', 'pc_dc199113-7c34-4cc2-85cb-e9b552b87b05.png', 'web_ff1666cb-81cb-4180-94ee-661a111749e4.png', 'pc_2086f81a-7a9f-472f-bfe7-50394f5a2bc6.png', 'web_b5c8e65b-1d1e-4fad-8064-a3b7119d1eaa.png', 'web_acad6928-ac22-40c7-98df-4e0df7b318bc.png', 'pc_83462e63-b0cd-4921

In [24]:
from pathlib import Path
import os

IMAGES_ROOT = Path("/home/ui_detect/DL_UI_Detection/ui_detection_ym/dataset/screenspot/images").resolve()

def fname_from_path(p: str) -> str:
    return Path(p.replace("\\", "/")).name

# 현재 images 아래 실제 파일명 set
existing = set([p.name for p in IMAGES_ROOT.rglob("*.png")])

total = 0
matched = 0
unmatched_examples = []

for rec in read_jsonl(INPUT_JSONL):
    total += 1
    fname = fname_from_path(rec["image"])
    if fname in existing:
        matched += 1
    elif len(unmatched_examples) < 5:
        unmatched_examples.append(fname)

print("total lines:", total)
print("matched:", matched)
print("match ratio:", matched / total if total else 0)
print("unmatched examples:", unmatched_examples)

total lines: 7792
matched: 3847
match ratio: 0.49371149897330596
unmatched examples: ['pc_00b0c9f6-b24f-4bff-b492-12117b50e63b.png', 'pc_00b0c9f6-b24f-4bff-b492-12117b50e63b.png', 'pc_00b0c9f6-b24f-4bff-b492-12117b50e63b.png', 'pc_00b0c9f6-b24f-4bff-b492-12117b50e63b.png', 'pc_00b0c9f6-b24f-4bff-b492-12117b50e63b.png']


In [28]:
from pathlib import Path
import os, json
from PIL import Image
from tqdm import tqdm

IMAGES_ROOT = Path("/home/ui_detect/DL_UI_Detection/ui_detection_ym/dataset/screenspot/images").resolve()
existing = {p.name: str(p) for p in IMAGES_ROOT.rglob("*.png")}  # fname -> full path

OUT = Path("./out_icon_descriptions_matched_only.jsonl")

n_total = 0
n_matched = 0
n_written = 0

with open(OUT, "w", encoding="utf-8") as w:
    for rec in tqdm(read_jsonl(INPUT_JSONL), desc="generating (matched only)"):
        n_total += 1
        fname = Path(rec["image"].replace("\\", "/")).name
        if fname not in existing:
            continue

        n_matched += 1
        img_path = existing[fname]

        img = Image.open(img_path).convert("RGB")
        box, icon_img, ctx_img = make_icon_and_context_crops(img, rec["bbox_xyxy"], ICON_PAD_PX, CTX_PAD_SCALE)
        desc = describe_icon(icon_img, ctx_img)

        out = dict(rec)
        out["image"] = img_path
        out["bbox_xyxy"] = box
        out["description"] = desc

        w.write(json.dumps(out, ensure_ascii=False) + "\n")
        n_written += 1

print("total:", n_total, "matched:", n_matched, "written:", n_written)
print("saved:", OUT)

generating (matched only): 7792it [1:59:49,  1.08it/s]

total: 7792 matched: 3847 written: 3847
saved: out_icon_descriptions_matched_only.jsonl
